In [2]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 14.0 MB/s  0:00:05m0:00:0100:01


In [ ]:
# ============================================================
# TRANSFORMER HYPERPARAMETER TUNING + FINAL FIGURES
# ============================================================

# import os, json, itertools
# import numpy as np
# import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# from sklearn.model_selection import KFold, train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import os
import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from torch.utils.data import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WEIGHT_DECAY = 0.0
MLP_HIDDEN = [256, 128, 64]
MLP_DROPOUT = 0.1


class MultiInputDataset(Dataset):
    def __init__(self, X_exp, X_cov, Y):
        self.X_exp = torch.tensor(X_exp, dtype=torch.float32)
        self.X_cov = torch.tensor(X_cov, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return self.Y.shape[0]

    def __getitem__(self, idx):
        return self.X_exp[idx], self.X_cov[idx], self.Y[idx]


class AttnEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True
        )
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, attn_w = self.mha(
            x, x, x,
            need_weights=True,
            average_attn_weights=False
        )
        x = self.norm1(x + self.drop(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.drop(ff_out))
        return x, attn_w


class MultiTaskTabularTransformer(nn.Module):
    def __init__(self, n_exposures, n_covariates, n_tasks,
                 embed_dim=32, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.n_covariates = n_covariates
        self.feature_embedding = nn.Linear(1, embed_dim)

        self.layers = nn.ModuleList([
            AttnEncoderLayer(embed_dim, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.regressor = nn.Sequential(
            nn.Linear(embed_dim + n_covariates, 64),
            nn.ReLU(),
            nn.Linear(64, n_tasks)
        )

    def forward(self, x_exp, x_cov):
        x = x_exp.unsqueeze(-1)
        x = self.feature_embedding(x)

        attn_layers = []
        for layer in self.layers:
            x, attn_w = layer(x)
            attn_layers.append(attn_w)

        pooled = x.mean(dim=1)

        if self.n_covariates > 0:
            pooled = torch.cat([pooled, x_cov], dim=1)

        yhat = self.regressor(pooled)
        return yhat, attn_layers


class MultiTaskMLP(nn.Module):
    def __init__(self, n_exposures, n_covariates, n_tasks,
                 hidden_sizes=(256, 128, 64), dropout=0.1):
        super().__init__()

        in_dim = n_exposures + n_covariates
        layers = []

        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h

        self.trunk = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, n_tasks)

    def forward(self, x_exp, x_cov):
        x = torch.cat([x_exp, x_cov], dim=1)
        z = self.trunk(x)
        return self.head(z)


def multitask_loss_mse(yhat, ytrue, weights=None):
    per_task_mse = (yhat - ytrue) ** 2
    per_task_mse = per_task_mse.mean(dim=0)

    if weights is None:
        return per_task_mse.mean()

    return (per_task_mse * weights).sum() / (weights.sum() + 1e-12)


def evaluate_model(model, test_loader, outcome_names, is_transformer=False):
    model.eval()

    y_true_list = []
    y_pred_list = []
    all_attn = None
    n_batches = 0

    with torch.no_grad():
        for Xexp_b, Xcov_b, Yb in test_loader:
            Xexp_b = Xexp_b.to(device)
            Xcov_b = Xcov_b.to(device)

            if is_transformer:
                Yhat, attn_layers = model(Xexp_b, Xcov_b)
                attn_stack = torch.stack(attn_layers, dim=0)
                attn_avg = attn_stack.mean(dim=(0, 1, 2))
                attn_np = attn_avg.cpu().numpy()
                all_attn = attn_np if all_attn is None else all_attn + attn_np
                n_batches += 1
            else:
                Yhat = model(Xexp_b, Xcov_b)

            y_true_list.append(Yb.numpy())
            y_pred_list.append(Yhat.cpu().numpy())

    y_true_all = np.vstack(y_true_list)
    y_pred_all = np.vstack(y_pred_list)

    metrics = {}
    for j, name in enumerate(outcome_names):
        metrics[name] = {
            "RMSE": float(np.sqrt(mean_squared_error(y_true_all[:, j], y_pred_all[:, j]))),
            "MAE": float(mean_absolute_error(y_true_all[:, j], y_pred_all[:, j])),
            "R2": float(r2_score(y_true_all[:, j], y_pred_all[:, j]))
        }

    if is_transformer and all_attn is not None:
        all_attn = all_attn / float(n_batches)

    return y_true_all, y_pred_all, metrics, all_attn


def train_model(model, train_loader, epochs=100, lr=1e-3,
                weight_decay=0.0, is_transformer=False, task_weights=None):
    model.to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    for epoch in range(epochs):
        model.train()

        for Xexp_b, Xcov_b, Yb in train_loader:
            Xexp_b = Xexp_b.to(device)
            Xcov_b = Xcov_b.to(device)
            Yb = Yb.to(device)

            optimizer.zero_grad()

            if is_transformer:
                Yhat, _ = model(Xexp_b, Xcov_b)
            else:
                Yhat = model(Xexp_b, Xcov_b)

            loss = multitask_loss_mse(Yhat, Yb, weights=task_weights)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 20 == 0:
            print(f"    Epoch {epoch+1}/{epochs} complete")

def clean_XY_multitask(X, Y, outcome_names, save_dir):
    """
    Clean X and Y for multi-task regression.
    - Replace inf with NaN
    - Impute X NaNs with column medians
    - Drop rows where any outcome is missing/non-finite
    - Drop rows where X is still non-finite
    """
    import os, json
    import numpy as np

    os.makedirs(save_dir, exist_ok=True)

    X = X.copy().astype(float)
    Y = Y.copy().astype(float)

    # Replace inf with NaN
    X[~np.isfinite(X)] = np.nan
    Y[~np.isfinite(Y)] = np.nan

    # Median imputation for X
    col_medians = np.nanmedian(X, axis=0)
    col_medians = np.where(np.isfinite(col_medians), col_medians, 0.0)

    rows, cols = np.where(np.isnan(X))
    if len(rows) > 0:
        X[rows, cols] = col_medians[cols]

    # Keep rows with valid X and all valid outcomes
    keep = np.isfinite(X).all(axis=1) & np.isfinite(Y).all(axis=1)

    report = {
        "outcomes": outcome_names,
        "rows_total": int(Y.shape[0]),
        "rows_kept": int(keep.sum()),
        "rows_dropped": int((~keep).sum())
    }

    with open(os.path.join(save_dir, "data_cleaning_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    return X[keep], Y[keep]
DATA_PATH = "/Users/doreennoldajehu-appiah/Desktop/DLL_Project/Data_NHANES.csv"
df = pd.read_csv(DATA_PATH)
# -----------------------------
# 1) TUNING CONFIG
# -----------------------------
TUNE_DIR = "/Users/doreennoldajehu-appiah/Desktop/DLL_Project/OUT"
os.makedirs(TUNE_DIR, exist_ok=True)

PARAM_GRID = {
    "embed_dim": [16, 32, 64],
    "n_heads": [2, 4],
    "n_layers": [1, 2, 3],
    "dropout": [0.05, 0.10, 0.20],
    "lr": [1e-4, 5e-4, 1e-3],
    "batch_size": [32, 64]
}

MAX_EPOCHS = 100
PATIENCE = 10
VAL_SIZE = 0.15
N_SPLITS = 5
RANDOM_STATE = 42

# -----------------------------
# 2) BUILD PARAMETER COMBINATIONS
# -----------------------------
all_configs = list(itertools.product(
    PARAM_GRID["embed_dim"],
    PARAM_GRID["n_heads"],
    PARAM_GRID["n_layers"],
    PARAM_GRID["dropout"],
    PARAM_GRID["lr"],
    PARAM_GRID["batch_size"]
))

# Optional: limit search for speed
MAX_CONFIGS = 20
np.random.seed(RANDOM_STATE)
if len(all_configs) > MAX_CONFIGS:
    selected_idx = np.random.choice(len(all_configs), MAX_CONFIGS, replace=False)
    all_configs = [all_configs[i] for i in selected_idx]

print(f"[INFO] Number of Transformer configs to test: {len(all_configs)}")


# -----------------------------
# DEFINE FEATURES
# -----------------------------

# Covariates
COVARIATE_COLS = ["age","Gender","Ethnicity","BMI","smoke","alcohol","income","Education"]
COVARIATE_COLS = [c for c in COVARIATE_COLS if c in df.columns]

# Outcomes
OUTCOME_COLS = ["FLI","ALT","AST","GGT","ALP","TotalBilirubin"]
OUTCOME_COLS = [c for c in OUTCOME_COLS if c in df.columns]

# Drop columns
DROP_FROM_FEATURES = ["seqn","sdmvpsu","sdmvstra","wtmec2yr"]

# Exposure features
exposure_cols = [
    c for c in df.columns
    if c not in OUTCOME_COLS
    and c not in DROP_FROM_FEATURES
    and c not in COVARIATE_COLS
]

# Build matrices
X_exposure_all = df[exposure_cols].values.astype(float)
X_covariate_all = df[COVARIATE_COLS].values.astype(float) if len(COVARIATE_COLS) > 0 else None
Y_all = df[OUTCOME_COLS].values.astype(float)
# -----------------------------
# 3) CLEAN DATA ONCE
# -----------------------------
if X_covariate_all is None:
    X_for_clean = X_exposure_all
else:
    X_for_clean = np.hstack([X_exposure_all, X_covariate_all])

X_clean, Y_clean = clean_XY_multitask(
    X_for_clean,
    Y_all,
    OUTCOME_COLS,
    os.path.join(TUNE_DIR, "data_cleaning")
)

n_exp = X_exposure_all.shape[1]
X_exp_clean = X_clean[:, :n_exp]
X_cov_clean = X_clean[:, n_exp:] if X_covariate_all is not None else np.zeros((len(Y_clean), 0))

print("[INFO] Clean data shape:", X_exp_clean.shape, X_cov_clean.shape, Y_clean.shape)

# -----------------------------
# 4) EARLY STOPPING TRAIN FUNCTION
# -----------------------------
def train_transformer_early_stop(
    model,
    train_loader,
    val_loader,
    lr=1e-3,
    max_epochs=100,
    patience=10
):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = np.inf
    best_state = None
    wait = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0

        for Xexp_b, Xcov_b, Yb in train_loader:
            Xexp_b = Xexp_b.to(device)
            Xcov_b = Xcov_b.to(device)
            Yb = Yb.to(device)

            optimizer.zero_grad()
            Yhat, _ = model(Xexp_b, Xcov_b)
            loss = multitask_loss_mse(Yhat, Yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= max(1, len(train_loader))

        # validation loss
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for Xexp_b, Xcov_b, Yb in val_loader:
                Xexp_b = Xexp_b.to(device)
                Xcov_b = Xcov_b.to(device)
                Yb = Yb.to(device)

                Yhat, _ = model(Xexp_b, Xcov_b)
                loss = multitask_loss_mse(Yhat, Yb)
                val_loss += loss.item()

        val_loss /= max(1, len(val_loader))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_val_loss, epoch + 1

# -----------------------------
# 5) TUNE TRANSFORMER
# -----------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

tuning_rows = []
best_config = None
best_score = np.inf

for config_id, config in enumerate(all_configs, start=1):

    embed_dim, n_heads, n_layers, dropout, lr, batch_size = config

    # skip invalid head/embedding combos
    if embed_dim % n_heads != 0:
        continue

    print("\n" + "="*70)
    print(f"[CONFIG {config_id}] embed={embed_dim}, heads={n_heads}, layers={n_layers}, "
          f"dropout={dropout}, lr={lr}, batch={batch_size}")
    print("="*70)

    fold_rmse_values = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_exp_clean), start=1):

        Xexp_train_raw, Xexp_test_raw = X_exp_clean[train_idx], X_exp_clean[test_idx]
        Xcov_train_raw, Xcov_test_raw = X_cov_clean[train_idx], X_cov_clean[test_idx]
        Y_train_full, Y_test = Y_clean[train_idx], Y_clean[test_idx]

        # validation split from training fold
        tr_idx, val_idx = train_test_split(
            np.arange(len(Y_train_full)),
            test_size=VAL_SIZE,
            random_state=RANDOM_STATE + fold,
            shuffle=True
        )

        Xexp_tr_raw, Xexp_val_raw = Xexp_train_raw[tr_idx], Xexp_train_raw[val_idx]
        Xcov_tr_raw, Xcov_val_raw = Xcov_train_raw[tr_idx], Xcov_train_raw[val_idx]
        Y_tr, Y_val = Y_train_full[tr_idx], Y_train_full[val_idx]

        # scale exposures
        scaler_exp = StandardScaler()
        Xexp_tr = scaler_exp.fit_transform(Xexp_tr_raw)
        Xexp_val = scaler_exp.transform(Xexp_val_raw)
        Xexp_test = scaler_exp.transform(Xexp_test_raw)

        # scale covariates
        if Xcov_train_raw.shape[1] > 0:
            scaler_cov = StandardScaler()
            Xcov_tr = scaler_cov.fit_transform(Xcov_tr_raw)
            Xcov_val = scaler_cov.transform(Xcov_val_raw)
            Xcov_test = scaler_cov.transform(Xcov_test_raw)
        else:
            Xcov_tr, Xcov_val, Xcov_test = Xcov_tr_raw, Xcov_val_raw, Xcov_test_raw

        train_loader = DataLoader(
            MultiInputDataset(Xexp_tr, Xcov_tr, Y_tr),
            batch_size=batch_size,
            shuffle=True
        )

        val_loader = DataLoader(
            MultiInputDataset(Xexp_val, Xcov_val, Y_val),
            batch_size=batch_size,
            shuffle=False
        )

        test_loader = DataLoader(
            MultiInputDataset(Xexp_test, Xcov_test, Y_test),
            batch_size=batch_size,
            shuffle=False
        )

        model = MultiTaskTabularTransformer(
            n_exposures=Xexp_tr.shape[1],
            n_covariates=Xcov_tr.shape[1],
            n_tasks=len(OUTCOME_COLS),
            embed_dim=embed_dim,
            n_heads=n_heads,
            n_layers=n_layers,
            dropout=dropout
        )

        best_val_loss, epochs_used = train_transformer_early_stop(
            model,
            train_loader,
            val_loader,
            lr=lr,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE
        )

        Y_true, Y_pred, metrics, attn = evaluate_model(
            model,
            test_loader,
            OUTCOME_COLS,
            is_transformer=True
        )

        mean_rmse = np.mean([metrics[o]["RMSE"] for o in OUTCOME_COLS])
        fold_rmse_values.append(mean_rmse)

        tuning_rows.append({
            "config_id": config_id,
            "fold": fold,
            "embed_dim": embed_dim,
            "n_heads": n_heads,
            "n_layers": n_layers,
            "dropout": dropout,
            "lr": lr,
            "batch_size": batch_size,
            "epochs_used": epochs_used,
            "val_loss": best_val_loss,
            "mean_fold_RMSE": mean_rmse
        })

        print(f"Fold {fold}: mean RMSE={mean_rmse:.4f}, epochs={epochs_used}")

    config_score = np.mean(fold_rmse_values)

    print(f"[CONFIG {config_id}] Mean CV RMSE = {config_score:.4f}")

    if config_score < best_score:
        best_score = config_score
        best_config = {
            "config_id": config_id,
            "embed_dim": embed_dim,
            "n_heads": n_heads,
            "n_layers": n_layers,
            "dropout": dropout,
            "lr": lr,
            "batch_size": batch_size,
            "mean_cv_rmse": best_score
        }

# -----------------------------
# 6) SAVE TUNING RESULTS
# -----------------------------
tuning_df = pd.DataFrame(tuning_rows)
tuning_path = os.path.join(TUNE_DIR, "transformer_tuning_results_by_fold.csv")
tuning_df.to_csv(tuning_path, index=False)

summary_tuning = (
    tuning_df.groupby(["config_id", "embed_dim", "n_heads", "n_layers", "dropout", "lr", "batch_size"])
    .agg(
        mean_RMSE=("mean_fold_RMSE", "mean"),
        std_RMSE=("mean_fold_RMSE", "std"),
        mean_epochs=("epochs_used", "mean")
    )
    .reset_index()
    .sort_values("mean_RMSE")
)

summary_path = os.path.join(TUNE_DIR, "transformer_tuning_summary.csv")
summary_tuning.to_csv(summary_path, index=False)

best_path = os.path.join(TUNE_DIR, "best_transformer_config.json")
with open(best_path, "w") as f:
    json.dump(best_config, f, indent=2)

print("\n[INFO] Best Transformer config:")
print(best_config)
print("[SAVED]", tuning_path)
print("[SAVED]", summary_path)
print("[SAVED]", best_path)

# -----------------------------
# 7) PLOT TUNING RESULTS
# -----------------------------
plt.figure(figsize=(10, 5))
plt.bar(summary_tuning["config_id"].astype(str), summary_tuning["mean_RMSE"],
        yerr=summary_tuning["std_RMSE"], capsize=4)
plt.xlabel("Transformer Config ID")
plt.ylabel("Mean RMSE across folds")
plt.title("Transformer Hyperparameter Tuning Results")
plt.xticks(rotation=90)
plt.tight_layout()

fig_path = os.path.join(TUNE_DIR, "fig_transformer_tuning_rmse.png")
plt.savefig(fig_path, dpi=300)
plt.close()
print("[SAVED]", fig_path)

# -----------------------------
# 8) FINAL TRAINING USING BEST CONFIG + MLP BASELINE
# -----------------------------
FINAL_DIR = os.path.join(TUNE_DIR, "final_best_model_run")
os.makedirs(FINAL_DIR, exist_ok=True)

best_embed = best_config["embed_dim"]
best_heads = best_config["n_heads"]
best_layers = best_config["n_layers"]
best_dropout = best_config["dropout"]
best_lr = best_config["lr"]
best_batch = best_config["batch_size"]

final_metrics_rows = []
final_predictions_rows = []
final_attn_sum = None
final_attn_folds = 0

for fold, (train_idx, test_idx) in enumerate(kf.split(X_exp_clean), start=1):

    print("\n" + "="*70)
    print(f"[FINAL FOLD {fold}/{N_SPLITS}]")
    print("="*70)

    fold_dir = os.path.join(FINAL_DIR, f"fold={fold}")
    os.makedirs(fold_dir, exist_ok=True)

    Xexp_train_raw, Xexp_test_raw = X_exp_clean[train_idx], X_exp_clean[test_idx]
    Xcov_train_raw, Xcov_test_raw = X_cov_clean[train_idx], X_cov_clean[test_idx]
    Y_train, Y_test = Y_clean[train_idx], Y_clean[test_idx]

    scaler_exp = StandardScaler()
    Xexp_train = scaler_exp.fit_transform(Xexp_train_raw)
    Xexp_test = scaler_exp.transform(Xexp_test_raw)

    if Xcov_train_raw.shape[1] > 0:
        scaler_cov = StandardScaler()
        Xcov_train = scaler_cov.fit_transform(Xcov_train_raw)
        Xcov_test = scaler_cov.transform(Xcov_test_raw)
    else:
        Xcov_train, Xcov_test = Xcov_train_raw, Xcov_test_raw

    train_loader = DataLoader(
        MultiInputDataset(Xexp_train, Xcov_train, Y_train),
        batch_size=best_batch,
        shuffle=True
    )

    test_loader = DataLoader(
        MultiInputDataset(Xexp_test, Xcov_test, Y_test),
        batch_size=best_batch,
        shuffle=False
    )

    # tuned transformer
    tfm = MultiTaskTabularTransformer(
        n_exposures=Xexp_train.shape[1],
        n_covariates=Xcov_train.shape[1],
        n_tasks=len(OUTCOME_COLS),
        embed_dim=best_embed,
        n_heads=best_heads,
        n_layers=best_layers,
        dropout=best_dropout
    )

    train_model(
        tfm,
        train_loader,
        epochs=MAX_EPOCHS,
        lr=best_lr,
        weight_decay=WEIGHT_DECAY,
        is_transformer=True,
        task_weights=None
    )

    Yt_true, Yt_pred, tfm_metrics, attn = evaluate_model(
        tfm,
        test_loader,
        OUTCOME_COLS,
        is_transformer=True
    )

    if attn is not None:
        pd.DataFrame(attn, index=exposure_cols, columns=exposure_cols).to_csv(
            os.path.join(fold_dir, "attention_matrix.csv")
        )
        final_attn_sum = attn if final_attn_sum is None else final_attn_sum + attn
        final_attn_folds += 1

    # MLP baseline
    mlp = MultiTaskMLP(
        n_exposures=Xexp_train.shape[1],
        n_covariates=Xcov_train.shape[1],
        n_tasks=len(OUTCOME_COLS),
        hidden_sizes=tuple(MLP_HIDDEN),
        dropout=MLP_DROPOUT
    )

    train_model(
        mlp,
        train_loader,
        epochs=MAX_EPOCHS,
        lr=best_lr,
        weight_decay=WEIGHT_DECAY,
        is_transformer=False,
        task_weights=None
    )

    Ym_true, Ym_pred, mlp_metrics, _ = evaluate_model(
        mlp,
        test_loader,
        OUTCOME_COLS,
        is_transformer=False
    )

    # save predictions
    merged = pd.DataFrame({"fold": np.full(len(Yt_true), fold, dtype=int)})
    for j, outcome in enumerate(OUTCOME_COLS):
        merged[f"y_true_{outcome}"] = Yt_true[:, j]
        merged[f"y_pred_transformer_{outcome}"] = Yt_pred[:, j]
        merged[f"y_pred_mlp_{outcome}"] = Ym_pred[:, j]

        final_predictions_rows.append(pd.DataFrame({
            "fold": fold,
            "model": "Tuned_Transformer",
            "outcome": outcome,
            "y_true": Yt_true[:, j],
            "y_pred": Yt_pred[:, j]
        }))

        final_predictions_rows.append(pd.DataFrame({
            "fold": fold,
            "model": "MLP",
            "outcome": outcome,
            "y_true": Ym_true[:, j],
            "y_pred": Ym_pred[:, j]
        }))

    merged.to_csv(os.path.join(fold_dir, "predictions_merged.csv"), index=False)

    # save metrics
    for outcome in OUTCOME_COLS:
        row = {"fold": fold, "model": "Tuned_Transformer", "outcome": outcome}
        row.update(tfm_metrics[outcome])
        final_metrics_rows.append(row)

        row = {"fold": fold, "model": "MLP", "outcome": outcome}
        row.update(mlp_metrics[outcome])
        final_metrics_rows.append(row)

# -----------------------------
# 9) SAVE FINAL METRICS + PREDICTIONS
# -----------------------------
final_metrics = pd.DataFrame(final_metrics_rows)
final_metrics_path = os.path.join(FINAL_DIR, "FINAL_metrics_by_fold.csv")
final_metrics.to_csv(final_metrics_path, index=False)

final_summary = (
    final_metrics.groupby(["model", "outcome"])
    .agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std")
    )
    .reset_index()
)

final_summary_path = os.path.join(FINAL_DIR, "FINAL_metrics_summary.csv")
final_summary.to_csv(final_summary_path, index=False)

final_predictions = pd.concat(final_predictions_rows, ignore_index=True)
final_predictions_path = os.path.join(FINAL_DIR, "FINAL_predictions_long.csv")
final_predictions.to_csv(final_predictions_path, index=False)

print("[SAVED]", final_metrics_path)
print("[SAVED]", final_summary_path)
print("[SAVED]", final_predictions_path)

# -----------------------------
# 10) SAVE AVERAGED ATTENTION
# -----------------------------
if final_attn_sum is not None and final_attn_folds > 0:
    final_attn_avg = final_attn_sum / final_attn_folds
    attn_path = os.path.join(FINAL_DIR, "FINAL_attention_matrix_avg.csv")
    pd.DataFrame(final_attn_avg, index=exposure_cols, columns=exposure_cols).to_csv(attn_path)
    print("[SAVED]", attn_path)

# -----------------------------
# 11) FINAL FIGURES LIKE YOUR SLIDE
# -----------------------------
FIG_DIR = os.path.join(FINAL_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

outcomes = list(final_summary["outcome"].unique())
models = list(final_summary["model"].unique())

# RMSE bar plot
x = np.arange(len(outcomes))
width = 0.35

plt.figure(figsize=(10, 5))
for i, model in enumerate(models):
    sub = final_summary[final_summary["model"] == model].set_index("outcome").reindex(outcomes)
    plt.bar(
        x + i * width,
        sub["RMSE_mean"],
        width,
        yerr=sub["RMSE_std"],
        capsize=4,
        label=model
    )

plt.xticks(x + width / 2, outcomes)
plt.ylabel("RMSE")
plt.title("RMSE Comparison Across Outcomes (mean ± std across folds)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_RMSE_comparison.png"), dpi=300)
plt.close()

# R2 bar plot
plt.figure(figsize=(10, 5))
for i, model in enumerate(models):
    sub = final_summary[final_summary["model"] == model].set_index("outcome").reindex(outcomes)
    plt.bar(
        x + i * width,
        sub["R2_mean"],
        width,
        yerr=sub["R2_std"],
        capsize=4,
        label=model
    )

plt.xticks(x + width / 2, outcomes)
plt.ylabel("R²")
plt.title("R² Comparison Across Outcomes (mean ± std across folds)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_R2_comparison.png"), dpi=300)
plt.close()

# Overall mean RMSE
overall = (
    final_metrics.groupby(["model", "fold"])
    .agg(mean_RMSE=("RMSE", "mean"))
    .reset_index()
)

overall_summary = (
    overall.groupby("model")
    .agg(mean_RMSE=("mean_RMSE", "mean"),
         std_RMSE=("mean_RMSE", "std"))
    .reset_index()
)

plt.figure(figsize=(6, 5))
plt.bar(
    overall_summary["model"],
    overall_summary["mean_RMSE"],
    yerr=overall_summary["std_RMSE"],
    capsize=5
)
plt.ylabel("Mean RMSE across outcomes")
plt.title("Overall Model Performance (mean ± std across folds)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_overall_model_performance.png"), dpi=300)
plt.close()

print("\n✅ Hyperparameter tuning complete.")
print("[BEST CONFIG]", best_config)
print("[FINAL RESULTS SAVED TO]", FINAL_DIR)

[INFO] Number of Transformer configs to test: 20
[INFO] Clean data shape: (4253, 39) (4253, 8) (4253, 6)

[CONFIG 1] embed=32, heads=2, layers=2, dropout=0.1, lr=0.0001, batch=32
Fold 1: mean RMSE=24.1466, epochs=100
